# 03_ Feature engineering


In this section, we perform feature engineering to transform raw variables into more informative features that improve model performance and interpretability.

Feature engineering helps:

- Capture hidden relationships in the data
- Improve predictive power
- Reduce noise
- Make variables more suitable for machine learning algorithms

***Import Required Libraries***

In [85]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder,MinMaxScaler,StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
sns.set_style('whitegrid')

## Loading the data

In [86]:
DATA_PATH = "../data/processed/dataset_clean.csv"
anime_df = pd.read_csv(DATA_PATH)
anime_df.head()

,name,genre,type,episodes,rating,members,missing_episodes,log_members,log_episodes
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural",movie,1.0,9.37,200630.0,0,12.209223,0.693147
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",tv,64.0,9.26,793665.0,0,13.584418,4.174387
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",tv,51.0,9.25,114262.0,0,11.646258,3.951244
3,Steins;Gate,"Sci-Fi, Thriller",tv,24.0,9.17,673572.0,0,13.420352,3.218876
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",tv,51.0,9.16,151266.0,0,11.926802,3.951244


In [87]:
anime_df['type'].unique()

array(['movie', 'tv', 'ova', 'special', 'music', 'ona'], dtype=object)

In [88]:
anime_df['genre'].unique()

array(['Drama, Romance, School, Supernatural',
       'Action, Adventure, Drama, Fantasy, Magic, Military, Shounen',
       'Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen',
       ..., 'Hentai, Sports', 'Drama, Romance, School, Yuri',
       'Hentai, Slice of Life'], dtype=object)

# Numeric features
now we scale the numeric features and standerize them 

In [89]:
scaler = StandardScaler()
num_cols = ['log_episodes', 'log_members','rating']
anime_df[num_cols] = scaler.fit_transform(anime_df[num_cols])

# Categorical features
we encode categorical features 

In [90]:
type_dummies = pd.get_dummies(anime_df['type'], prefix='type')
anime_df = pd.concat([anime_df,type_dummies], axis = 1)
anime_df= anime_df.drop(columns=['type'])

In [91]:
display(anime_df.head())

,name,genre,episodes,rating,members,missing_episodes,log_members,log_episodes,type_movie,type_music,type_ona,type_ova,type_special,type_tv
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural",1.0,2.845534,200630.0,0,2.060713,-0.804199,True,False,False,False,False,False
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",64.0,2.737388,793665.0,0,2.653134,2.134370,False,False,False,False,False,True
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",51.0,2.727556,114262.0,0,1.818193,1.946011,False,False,False,False,False,True
3,Steins;Gate,"Sci-Fi, Thriller",24.0,2.648904,673572.0,0,2.582456,1.327808,False,False,False,False,False,True
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",51.0,2.639073,151266.0,0,1.939049,1.946011,False,False,False,False,False,True


for the genre and since there is a lot of it we will deal with it that way
- Count genre frequencies
- Keep only Top N genres
- Remove rare genres
- Apply MultiLabelBinarizer

In [92]:
anime_df['genre_list'] = anime_df['genre'].apply(
    lambda x: [g.strip() for g in x.split(',')] if pd.notnull(x) else []
)

In [93]:
all_genres = [genre for sublist in anime_df['genre_list'] for genre in sublist]

genre_counts = Counter(all_genres)

genre_freq_df = pd.DataFrame(genre_counts.items(), columns=['Genre', 'Count'])
genre_freq_df = genre_freq_df.sort_values(by='Count', ascending=False)

print(genre_freq_df.head(20))

            Genre  Count
10         Comedy   4645
4          Action   2845
5       Adventure   2348
6         Fantasy   2309
14         Sci-Fi   2070
0           Drama   2016
9         Shounen   1712
38           Kids   1609
1         Romance   1464
2          School   1220
19  Slice of Life   1220
40         Hentai   1141
3    Supernatural   1037
20          Mecha    944
21          Music    860
11     Historical    806
7           Magic    778
31          Ecchi    637
26         Shoujo    603
23         Seinen    547


In [94]:
MIN_FREQ = 50
top_genres = [g for g, count in genre_counts.items() if count >= MIN_FREQ]

In [95]:
anime_df['filtered_genres'] = anime_df['genre_list'].apply(
    lambda genres: [g for g in genres if g in top_genres]
)


In [96]:

mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(anime_df['filtered_genres'])

genre_df = pd.DataFrame(
    genre_encoded,
    columns=[f'genre_{g}' for g in mlb.classes_]
)

anime_df = pd.concat([anime_df, genre_df], axis=1)


In [97]:
anime_df = anime_df.drop(columns=['genre', 'genre_list', 'filtered_genres'])

In [98]:
anime_df.to_csv("../data/processed/dataset_ready.csv", index=False)